# Análisis Exploratorio de Datos (EDA) - Airbnb Mexico


## Objetivo
Entender la estructura, calidad y características de los datos antes de realizar transformaciones.

**Colecciones a analizar:**
- `Listings`: Información detallada de los apartamentos
- `Reviews`: Información de reseñas de clientes
- `Calendar`: Información de disponibilidad

## Importaciones

In [8]:
import sys
from pathlib import Path

# Raíz del proyecto: sirve si el cwd es `notebooks/` o la raíz del repo
_cwd = Path.cwd().resolve()
if (_cwd / "src").is_dir():
    _root = _cwd
elif (_cwd.parent / "src").is_dir():
    _root = _cwd.parent
else:
    _root = _cwd.parent
sys.path.insert(0, str(_root))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import Extraccion, get_logger

# Configuración de visualizaciones
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

logger = get_logger(__name__)
print("✓ Importaciones completadas")

✓ Importaciones completadas


## Funciones comunes

In [9]:
import re

from IPython.display import display


def analizar_campo_price(df, col="price", nombre_coleccion="", muestra_ejemplos=8):
    """
    Análisis exploratorio del campo price para decidir transformaciones (moneda, separadores, texto).
    No modifica el DataFrame. Funciona con columnas numéricas u object/string.
    """
    titulo = f" - {nombre_coleccion}" if nombre_coleccion else ""
    print(f"\nANÁLISIS DE CAMPO `{col}`{titulo}")
    print("-" * 80)

    if col not in df.columns:
        print(f"   Columna '{col}' no existe.")
        return None

    serie = df[col]
    print(f"   Tipo pandas: {serie.dtype}")
    n_total = len(serie)
    n_nulos = int(serie.isna().sum())
    pct_nulos = 100 * n_nulos / n_total if n_total else 0
    print(f"   Registros: {n_total:,} | Nulos: {n_nulos:,} ({pct_nulos:.2f}%)")

    no_nulo = serie.dropna()
    if len(no_nulo) == 0:
        print("   No hay valores no nulos para analizar.")
        return {"resumen": "sin_datos"}

    num_coerced = pd.to_numeric(serie, errors="coerce")
    n_parse_ok = int(num_coerced.notna().sum())
    n_parse_fail = int((serie.notna() & num_coerced.isna()).sum())
    print(f"\n   pd.to_numeric(errors='coerce'):")
    print(f"      - Reconocidos como número: {n_parse_ok:,}")
    print(f"      - No nulos que NO convierten: {n_parse_fail:,}")

    str_vals = no_nulo.astype(str).str.strip()

    def _clasificar_formato(s):
        s = str(s).strip()
        if re.search(r"[\$€£¥¢]", s):
            return "simbolo_moneda_en_texto"
        if re.search(r"\b(MXN|USD|EUR|GBP|ARS|COP|CLP)\b", s, re.I):
            return "codigo_moneda_en_texto"
        if re.search(r"[A-Za-z]", s) and not re.match(r"^[-+]?[\d\s,\.eE]+$", s):
            return "texto_u_otro_no_numerico"
        limpio = s.replace(" ", "")
        if re.match(r"^-?\d+$", limpio):
            return "entero_sin_separador_decimal"
        if re.match(r"^-?\d{1,3}(,\d{3})+(\.\d+)?$", limpio):
            return "miles_con_coma_punto_decimal_opcional"
        if re.match(r"^-?\d{1,3}(\.\d{3})+(,\d+)?$", limpio):
            return "miles_con_punto_coma_decimal_opcional"
        if re.match(r"^-?\d+\.\d+$", limpio):
            return "punto_como_decimal"
        if re.match(r"^-?\d+,\d+$", limpio):
            return "coma_como_decimal"
        if re.match(r"^-?[\d\s,\.]+$", limpio):
            return "digitos_con_separadores_mixtos"
        return "otro"

    tiene_simbolo = str_vals.str.contains(r"[\$€£¥¢]", regex=True)
    patrones = str_vals.map(_clasificar_formato)
    dist = patrones.value_counts()
    print(f"\n   Formato de la representación en texto (no nulos):")
    display(dist.to_frame("conteo"))
    print(f"   Filas con símbolo $ € £ ¥ ¢ en el texto: {int(tiene_simbolo.sum()):,}")

    if n_parse_fail > 0:
        ejemplos = (
            serie[serie.notna() & num_coerced.isna()].drop_duplicates().head(muestra_ejemplos)
        )
        print(f"\n   Ejemplos que no convierten a número (hasta {muestra_ejemplos}):")
        display(ejemplos.to_frame(col))

    if num_coerced.notna().any():
        v = num_coerced.dropna()
        print(f"\n   Estadísticos si se interpretan como número:")
        print(f"      - Mín: {v.min()} | Máx: {v.max()} | Media: {v.mean():.6g}")
        neg = int((v < 0).sum())
        if neg:
            print(f"      - Negativos: {neg:,} (revisar con analizar_valores_negativos)")

    if pd.api.types.is_numeric_dtype(serie):
        print(f"\n   Nota: la columna ya es numérica; el análisis de 'formato texto' aplica al castear a string.")

    return {
        "dtype": str(serie.dtype),
        "n_parse_ok": n_parse_ok,
        "n_parse_fail": n_parse_fail,
        "distribucion_formato": dist.to_dict(),
    }


def analizar_valores_negativos(df, columnas, nombre_coleccion=""):
    """
    Indica si hay valores negativos en columnas (numéricas o convertibles).
    Imprime conteos, mínimo y máximo sobre valores que sí convierten a número.
    """
    titulo = f" - {nombre_coleccion}" if nombre_coleccion else ""
    print(f"\nVALORES NEGATIVOS{titulo}")
    print("-" * 80)

    resultado = {}
    for col in columnas:
        if col not in df.columns:
            print(f"\n   [{col}] No existe en el DataFrame.")
            resultado[col] = {"error": "columna_inexistente"}
            continue

        num = pd.to_numeric(df[col], errors="coerce")
        valid = num.dropna()
        n_valid = len(valid)
        n_neg = int((valid < 0).sum())
        n_cero = int((valid == 0).sum())
        n_pos = int((valid > 0).sum())
        n_no_convert = int(df[col].notna().sum() - n_valid)

        print(f"\n   Columna: {col}")
        print(f"      - Numéricos válidos: {n_valid:,} | No nulos no convertibles: {n_no_convert:,}")
        print(f"      - Negativos: {n_neg:,} | Ceros: {n_cero:,} | Positivos: {n_pos:,}")
        if n_valid > 0:
            print(f"      - Mínimo: {valid.min()} | Máximo: {valid.max()}")
        if n_neg > 0:
            print(f"      >>> Hay valores negativos: conviene regla de transformación o limpieza.")
        else:
            print(f"      >>> Sin valores negativos en valores numéricos válidos.")

        resultado[col] = {
            "n_negativos": n_neg,
            "n_validos": n_valid,
            "min": float(valid.min()) if n_valid else None,
            "max": float(valid.max()) if n_valid else None,
            "requiere_atencion": n_neg > 0,
        }
    return resultado


def analizar_estructura(df, nombre_coleccion):
    """
    Realiza análisis general de estructura de un DataFrame.
    
    Muestra:
    - Primeras filas
    - Dimensiones
    - Tipos de datos
    - Estadísticas numéricas
    - Estadísticas generales
    - Resumen de variables no numéricas
    """
    print("\n" + "=" * 80)
    print(f"COLECCIÓN: {nombre_coleccion}")
    print("=" * 80)
    
    print(f"\nPRIMERAS FILAS (primeros 5 registros):")
    display(df.head(5))
    
    print(f"\nDIMENSIONES:")
    print(f"   - Filas: {df.shape[0]:,}")
    print(f"   - Columnas: {df.shape[1]}")
    
    print(f"\nTIPOS DE DATOS:")
    print("INFO")
    df.info(verbose=False)
    print("TYPES")
    print(df.dtypes)
    
    print(f"\nESTADÍSTICAS (Columnas numéricas):")
    display(df.describe().T)
    
    print(f"\nESTADÍSTICAS COMPLETAS (incluye variables categóricas):")
    display(df.describe(include='all').T)
    
    cols_no_numericas = df.select_dtypes(include=['object', 'string', 'bool', 'datetime64', 'str']).columns
    if len(cols_no_numericas) > 0:
        print(f"\nCOLUMNAS NO NUMÉRICAS ({len(cols_no_numericas)}):")
        for col in cols_no_numericas:
            try:
                unicos = df[col].nunique(dropna=False)
                print(f"   - {col}: {unicos} valores únicos")
                top = df[col].value_counts(dropna=False).head(5)
                print(f"      Principales valores:\n{top.to_dict()}")
            except Exception:
                print(f"   - {col}: Contiene datos complejos o no se puede resumir")


def analizar_valores_nulos(df, nombre_coleccion):
    """
    Analiza valores nulos y faltantes en un DataFrame.
    
    Muestra:
    - Cantidad de nulos por columna
    - Porcentaje respecto al total de registros
    - Solo columnas que tienen al menos un nulo
    """
    print(f"\nVALORES NULOS Y FALTANTES - {nombre_coleccion}:")
    print("-" * 80)
    
    nulos = df.isnull().sum()
    total_registros = len(df)
    nulos_pct = (nulos / total_registros * 100).round(2)
    
    df_nulos = pd.DataFrame({
        'Columna': nulos.index,
        'Cantidad de nulos': nulos.values,
        'Porcentaje (%)': nulos_pct.values
    })
    
    # Filtrar solo columnas con nulos
    df_nulos_filtrado = df_nulos[df_nulos['Cantidad de nulos'] > 0].sort_values(
        'Cantidad de nulos', ascending=False
    )
    
    if len(df_nulos_filtrado) > 0:
        print(f"   Columnas con valores faltantes:")
        display(df_nulos_filtrado)
    else:
        print(f"   Todas las columnas están completas (sin nulos)")


def analizar_duplicados(df, nombre_coleccion, col_id='_id'):
    """
    Analiza registros duplicados de dos formas:
    1. Duplicados por columna ID (identidad unica)
    2. Filas completamente duplicadas en todas sus columnas.

    Si el DataFrame contiene listas o diccionarios, los convierte a una forma
    hashable para poder comparar filas completas sin errores.
    """
    print(f"\nREGISTROS DUPLICADOS - {nombre_coleccion}:")
    print("-" * 80)

    total = len(df)

    # Analisis 1: Duplicados por ID
    unicos = df[col_id].nunique()
    duplicados_id = total - unicos
    pct_duplicados_id = (duplicados_id / total * 100) if total > 0 else 0

    print(f"   1. POR COLUMNA ID ({col_id}):")
    print(f"      - Total de registros: {total:,}")
    print(f"      - Registros unicos: {unicos:,}")
    print(f"      - Duplicados por ID: {duplicados_id:,} ({pct_duplicados_id:.2f}%)")

    # Analisis 2: Filas completamente duplicadas
    def _hacer_hashable(valor):
        if isinstance(valor, list):
            return tuple(_hacer_hashable(v) for v in valor)
        if isinstance(valor, dict):
            return tuple(sorted((k, _hacer_hashable(v)) for k, v in valor.items()))
        if isinstance(valor, set):
            return tuple(sorted(_hacer_hashable(v) for v in valor))
        return valor

    try:
        filas_duplicadas = df.duplicated().sum()
        columnas_normalizadas = []
    except TypeError:
        df_hashable = df.copy()
        columnas_normalizadas = []

        columnas_objeto = df_hashable.select_dtypes(include=['object']).columns

        for col in columnas_objeto:
            columnas_normalizadas.append(col)
            df_hashable[col] = df_hashable[col].map(_hacer_hashable)

        filas_duplicadas = df_hashable.duplicated().sum()

    pct_filas_dup = (filas_duplicadas / total * 100) if total > 0 else 0

    print(f"\n   2. FILAS COMPLETAMENTE DUPLICADAS (todas las columnas):")
    print(f"      - Filas duplicadas: {filas_duplicadas:,} ({pct_filas_dup:.2f}%)")

    if columnas_normalizadas:
        print(
            "      - Nota: se normalizaron columnas con listas/dicts para comparar "
            f"filas completas: {columnas_normalizadas}"
        )

    if duplicados_id == 0 and filas_duplicadas == 0:
        print("\n   Excelente: No hay duplicados en esta coleccion")
def analizar_outliers(df, nombre_coleccion, columnas=None):
    """
    Analiza outliers en columnas numéricas usando método de cuartiles (IQR).
    """
    if columnas is None:
        columnas = df.select_dtypes(include='number').columns.tolist()
    
    print(f"\nANÁLISIS DE OUTLIERS - {nombre_coleccion}:")
    print("-" * 80)
    
    if len(columnas) == 0:
        print("   No hay columnas numéricas para analizar")
        return
    
    for col in columnas:
        if col not in df.columns:
            print(f"   Columna '{col}' no encontrada")
            continue
            
        datos = pd.to_numeric(df[col], errors='coerce').dropna()
        
        if len(datos) == 0:
            print(f"\n   {col}: No hay datos numéricos válidos")
            continue
        
        Q1 = datos.quantile(0.25)
        Q3 = datos.quantile(0.75)
        IQR = Q3 - Q1
        
        if IQR == 0:
            print(f"\n   {col}: IQR = 0, no hay variación suficiente para detectar outliers con este método")
            continue
        
        limite_inferior = Q1 - 1.5 * IQR
        limite_superior = Q3 + 1.5 * IQR
        
        outliers = datos[(datos < limite_inferior) | (datos > limite_superior)]
        pct_outliers = len(outliers) / len(datos) * 100
        
        print(f"\n   {col}:")
        print(f"      - Mínimo: {datos.min():.2f}")
        print(f"      - Q1 (25%): {Q1:.2f}")
        print(f"      - Mediana: {datos.median():.2f}")
        print(f"      - Q3 (75%): {Q3:.2f}")
        print(f"      - Máximo: {datos.max():.2f}")
        print(f"      - IQR: {IQR:.2f}")
        print(f"      - Outliers detectados: {len(outliers):,} ({pct_outliers:.2f}%)")
        print(f"      - Rango válido: [{limite_inferior:.2f}, {limite_superior:.2f}]")


## Extracción de datos desde MongoDB

In [10]:
# Conectar a MongoDB y extraer datos
extraccion = Extraccion()
print("Conectando a MongoDB...")
extraccion.conectar()

print("\nExtrayendo colecciones...")
df_calendar = extraccion.extraer_coleccion("Calendar", limit=100000)
df_listings = extraccion.extraer_coleccion("Listings", limit=100000)
df_reviews = extraccion.extraer_coleccion("Reviews", limit=100000)

print("\nExtracción completada")
print(f"\nRESUMEN:")
print(f"   - Calendar: {len(df_calendar):,} registros, {len(df_calendar.columns)} columnas")
print(f"   - Listings: {len(df_listings):,} registros, {len(df_listings.columns)} columnas")
print(f"   - Reviews: {len(df_reviews):,} registros, {len(df_reviews.columns)} columnas")


2026-04-12 23:25:30,681 | INFO | Extraccion | Conexion exitosa a MongoDB. Base de datos seleccionada: airbnb_mx


Conectando a MongoDB...

Extrayendo colecciones...


2026-04-12 23:25:34,918 | INFO | Extraccion | Coleccion Calendar extraida correctamente. Registros encontrados: 9873624. Registros cargados en DataFrame: 100000.
2026-04-12 23:25:36,695 | INFO | Extraccion | Coleccion Listings extraida correctamente. Registros encontrados: 27051. Registros cargados en DataFrame: 27051.
2026-04-12 23:25:38,083 | INFO | Extraccion | Coleccion Reviews extraida correctamente. Registros encontrados: 1454740. Registros cargados en DataFrame: 100000.



Extracción completada

RESUMEN:
   - Calendar: 100,000 registros, 6 columnas
   - Listings: 27,051 registros, 77 columnas
   - Reviews: 100,000 registros, 7 columnas


## 1. Análisis de Colección: LISTINGS

Información detallada sobre los apartamentos disponibles en Airbnb Buenos Aires.

### Análisis de estructura

In [11]:
analizar_estructura(df_listings, "LISTINGS")


COLECCIÓN: LISTINGS

PRIMERAS FILAS (primeros 5 registros):


,_id,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,...,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,reviews_per_month
0,69dc577b1d171e0e298f6505,35797,https://www.airbnb.com/rooms/35797,20250927041820,2025-09-27,city scrape,Villa Dante,"Dentro de Villa un estudio de arte con futon, ...","Santa Fe Shopping Mall, Interlomas Park and th...",https://a0.muscache.com/pictures/f395ab78-1185...,...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,69dc577b1d171e0e298f6506,44616,https://www.airbnb.com/rooms/44616,20250927041820,2025-09-28,city scrape,Condesa Haus,A new concept of hosting in mexico through a b...,NaN,https://a0.muscache.com/pictures/251410/ec75fe...,...,2011-11-09,2025-01-01,4.59,4.56,4.70,4.87,4.78,4.98,4.47,0.38
2,69dc577b1d171e0e298f6507,56074,https://www.airbnb.com/rooms/56074,20250927041820,2025-09-28,city scrape,Great space in historical San Rafael,This great apartment is located in one of the ...,Very traditional neighborhood with all service...,https://a0.muscache.com/pictures/3005118/60dac...,...,2011-04-02,2025-02-27,4.87,4.95,4.88,4.98,4.94,4.76,4.79,0.48
3,69dc577b1d171e0e298f6508,67703,https://www.airbnb.com/rooms/67703,20250927041820,2025-09-28,previous scrape,"2 bedroom apt. deco bldg, Condesa","Comfortably furnished, sunny, 2 bedroom apt., ...",NaN,https://a0.muscache.com/pictures/3281720/6f078...,...,2011-11-17,2024-10-30,4.90,4.82,4.76,4.94,4.92,4.98,4.92,0.30
4,69dc577b1d171e0e298f6509,70644,https://www.airbnb.com/rooms/70644,20250927041820,2025-09-28,previous scrape,Beautiful light Studio Coyoacan- full equipped !,COYOACAN designer studio quiet & safe! well eq...,Coyoacan is a beautiful neighborhood famous fo...,https://a0.muscache.com/pictures/f397d2da-d045...,...,2012-02-14,2025-08-18,4.91,4.90,4.96,4.96,4.98,4.96,4.92,0.81



DIMENSIONES:
   - Filas: 27,051
   - Columnas: 77

TIPOS DE DATOS:
INFO
<class 'pandas.DataFrame'>
RangeIndex: 27051 entries, 0 to 27050
Columns: 77 entries, _id to reviews_per_month
dtypes: bool(1), datetime64[us](5), float64(22), int64(20), object(9), str(20)
memory usage: 15.7+ MB
TYPES
_id                                       str
id                                      int64
listing_url                               str
scrape_id                               int64
last_scraped                   datetime64[us]
                                    ...      
review_scores_checkin                 float64
review_scores_communication           float64
review_scores_location                float64
review_scores_value                   float64
reviews_per_month                     float64
Length: 77, dtype: object

ESTADÍSTICAS (Columnas numéricas):


,count,mean,min,25%,50%,75%,max,std
id,27051.0,700355582609886464.0,35797.0,44104725.0,830736761677326976.0,1217382011952653056.0,1518561444972990720.0,570271415889610304.0
scrape_id,27051.0,20250927041820.0,20250927041820.0,20250927041820.0,20250927041820.0,20250927041820.0,20250927041820.0,0.0
last_scraped,27051,2025-09-27 17:33:41.381834,2025-09-27 00:00:00,2025-09-27 00:00:00,2025-09-28 00:00:00,2025-09-28 00:00:00,2025-09-28 00:00:00,NaN
host_id,27051.0,242943885.020554,7365.0,56214432.0,176683531.0,428882972.0,720844507.0,206257879.87984
host_since,27045,2018-10-02 05:48:54.708818,2009-02-03 00:00:00,2016-02-12 00:00:00,2018-03-05 00:00:00,2021-10-25 00:00:00,2025-09-25 00:00:00,NaN
host_listings_count,27045.0,24.604807,1.0,1.0,4.0,15.0,989.0,81.409257
host_total_listings_count,27045.0,33.77164,1.0,2.0,6.0,18.0,1196.0,116.146348
latitude,27051.0,19.40549,19.177848,19.392489,19.41539,19.432015,19.56101,0.04239
longitude,27051.0,-99.165385,-99.33963,-99.178499,-99.1671,-99.153731,-98.96336,0.033535
accommodates,27051.0,3.339285,1.0,2.0,2.0,4.0,16.0,2.344365



ESTADÍSTICAS COMPLETAS (incluye variables categóricas):


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
_id,27051,27051,69dc577b1d171e0e298f6505,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id,27051.0,NaN,NaN,NaN,700355582609886464.0,35797.0,44104725.0,830736761677326976.0,1217382011952653056.0,1518561444972990720.0,570271415889610304.0
listing_url,27051,27051,https://www.airbnb.com/rooms/35797,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
scrape_id,27051.0,NaN,NaN,NaN,20250927041820.0,20250927041820.0,20250927041820.0,20250927041820.0,20250927041820.0,20250927041820.0,0.0
last_scraped,27051,NaN,NaN,NaN,2025-09-27 17:33:41.381834,2025-09-27 00:00:00,2025-09-27 00:00:00,2025-09-28 00:00:00,2025-09-28 00:00:00,2025-09-28 00:00:00,NaN
...,...,...,...,...,...,...,...,...,...,...,...
review_scores_checkin,23649.0,NaN,NaN,NaN,4.836728,1.0,4.82,4.92,5.0,5.0,0.325204
review_scores_communication,23649.0,NaN,NaN,NaN,4.825281,1.0,4.81,4.92,5.0,5.0,0.355912
review_scores_location,23649.0,NaN,NaN,NaN,4.841777,1.0,4.81,4.92,5.0,5.0,0.297361
review_scores_value,23649.0,NaN,NaN,NaN,4.714353,1.0,4.67,4.8,4.91,5.0,0.391793



COLUMNAS NO NUMÉRICAS (35):
   - _id: 27051 valores únicos
      Principales valores:
{'69dc577b1d171e0e298f6505': 1, '69dc577b1d171e0e298f6506': 1, '69dc577b1d171e0e298f6507': 1, '69dc577b1d171e0e298f6508': 1, '69dc577b1d171e0e298f6509': 1}
   - listing_url: 27051 valores únicos
      Principales valores:
{'https://www.airbnb.com/rooms/35797': 1, 'https://www.airbnb.com/rooms/44616': 1, 'https://www.airbnb.com/rooms/56074': 1, 'https://www.airbnb.com/rooms/67703': 1, 'https://www.airbnb.com/rooms/70644': 1}
   - last_scraped: 2 valores únicos
      Principales valores:
{Timestamp('2025-09-28 00:00:00'): 19794, Timestamp('2025-09-27 00:00:00'): 7257}
   - source: 2 valores únicos
      Principales valores:
{'city scrape': 23567, 'previous scrape': 3484}
   - name: 25638 valores únicos
      Principales valores:
{'Blueground | Roma Sur 1 recamara, AC & rooftop': 52, 'Perfecto Loft en gran ubicación': 48, 'Blueground | Polanco, Luxury Apartment': 37, 'Blueground | Amueblado, Security & 

### Interpretación del resultado

- La colección Listings (Anuncios) corresponde a los anuncios que la gente publica en las cuales promociona sus propiedad, habitaciones, experiencias, etc.
- La colección tiene la siguiente estructura: 
    - Filas: 27.051
    - Columnas: 17

*Explicación de sus campos*
- **id**  
  Identificador único del listing en Airbnb.

- **name**  
  Nombre del alojamiento.

- **host_id**  
  Identificador único del anfitrión.

- **host_name**  
  Nombre del anfitrión.

- **neighbourhood**  
  Barrio o zona donde se encuentra el alojamiento.

- **latitude**  
  Latitud geográfica.

- **longitude**  
  Longitud geográfica.

- **room_type**  
  Tipo de propiedad:
  - `Entire home/apt`: alojamiento completo
  - `Private room`: habitación privada
  - `Shared room`: habitación compartida

- **minimum_nights**  
  Número mínimo de noches requeridas por reserva.

- **price**  
  Precio por noche (en MXN).

- **number_of_reviews**  
  Número total de reseñas recibidas.

- **last_review**  
  Fecha de la última reseña.

- **reviews_per_month**  
  Promedio de reseñas por mes.

- **number_of_reviews_ltm**  
  Número de reseñas en los últimos 12 meses.

- **calculated_host_listings_count**  
  Número de propiedades que tiene el anfitrión.

- **availability_365**  
  Número de días en el año en los que el alojamiento está disponible.

### Interpretación general

Este dataset permite analizar:
- Oferta de alojamientos
- Precios por zona
- Actividad y demanda (reseñas)
- Tipo de anfitriones (casuales vs profesionales)
- Disponibilidad anual




### Análisis de valores nulos

In [12]:
analizar_valores_nulos(df_listings, "LISTINGS")

# Coherencia last_review vs reviews_per_month (mismos nulos)
_mask = df_listings["last_review"].isna() | df_listings["reviews_per_month"].isna()
print("\nMUESTRA: last_review o reviews_per_month nulos (head 15)")
display(df_listings.loc[_mask].head(15))

_lr = df_listings["last_review"].isna()
_rpm = df_listings["reviews_per_month"].isna()
print("\nTabla de contingencia (filas nulas en cada columna):")
display(pd.crosstab(_lr, _rpm, rownames=["last_review_nulo"], colnames=["reviews_per_month_nulo"]))

ambos_igual = (_lr == _rpm).all()
solo_lr = int((_lr & ~_rpm).sum())
solo_rpm = int((~_lr & _rpm).sum())
print(f"\n¿Siempre coinciden los nulos? {ambos_igual}")
if not ambos_igual:
    print(f"   Filas con solo last_review nulo: {solo_lr}")
    print(f"   Filas con solo reviews_per_month nulo: {solo_rpm}")



VALORES NULOS Y FALTANTES - LISTINGS:
--------------------------------------------------------------------------------
   Columnas con valores faltantes:


,Columna,Cantidad de nulos,Porcentaje (%)
8,neighborhood_overview,13315,49.22
27,neighbourhood,13315,49.22
66,host_neighbourhood,12264,45.34
15,host_about,10829,40.03
14,host_location,5827,21.54
37,beds,3506,12.96
34,bathrooms,3496,12.92
60,estimated_revenue_l365d,3484,12.88
39,price,3484,12.88
72,review_scores_checkin,3402,12.58



MUESTRA: last_review o reviews_per_month nulos (head 15)


,_id,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,...,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,reviews_per_month
0,69dc577b1d171e0e298f6505,35797,https://www.airbnb.com/rooms/35797,20250927041820,2025-09-27,city scrape,Villa Dante,"Dentro de Villa un estudio de arte con futon, ...","Santa Fe Shopping Mall, Interlomas Park and th...",https://a0.muscache.com/pictures/f395ab78-1185...,...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,69dc577b1d171e0e298f650a,131610,https://www.airbnb.com/rooms/131610,20250927041820,2025-09-28,previous scrape,MARIA DEL ALMA,NaN,NaN,https://a0.muscache.com/pictures/837085/b9ed71...,...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,69dc577b1d171e0e298f6517,276504,https://www.airbnb.com/rooms/276504,20250927041820,2025-09-28,previous scrape,High End Condo with golf package,NaN,NaN,https://a0.muscache.com/pictures/2802432/4be14...,...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,69dc577b1d171e0e298f651a,291804,https://www.airbnb.com/rooms/291804,20250927041820,2025-09-28,previous scrape,A nice room with great location,NaN,NaN,https://a0.muscache.com/pictures/3323620/a48d3...,...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,69dc577b1d171e0e298f6520,331030,https://www.airbnb.com/rooms/331030,20250927041820,2025-09-27,city scrape,"Rento Recámara, Baño privado, Coyoacán Conchita","Wide bedroom, with private bathroom inside the...",Coyoacán is a beautiful and typical place of t...,https://a0.muscache.com/pictures/2f5aa915-f9b0...,...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,69dc577b1d171e0e298f6521,350079,https://www.airbnb.com/rooms/350079,20250927041820,2025-09-28,previous scrape,beautiful furnished room Rome area,NaN,NaN,https://a0.muscache.com/pictures/3875519/b11cd...,...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,69dc577b1d171e0e298f6536,642725,https://www.airbnb.com/rooms/642725,20250927041820,2025-09-28,previous scrape,Recamara en depa a 5 minutos del metropolitano,"north of the city, close to main square, resta...","north area of Gto lion, near Los Carcamos and ...",https://a0.muscache.com/pictures/e56d938e-c940...,...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100,69dc577b1d171e0e298f6569,1656382,https://www.airbnb.com/rooms/1656382,20250927041820,2025-09-28,previous scrape,Habitación individual c baño propio,"Private room with its own bathroom, spacious, ...",NaN,https://a0.muscache.com/pictures/23784927/519f...,...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
145,69dc577b1d171e0e298f6596,2516668,https://www.airbnb.com/rooms/2516668,20250927041820,2025-09-28,previous scrape,Room in new apartment of 1250ft²,I share one room in the central zone of Narvar...,NaN,https://a0.muscache.com/pictures/33261387/e058...,...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158,69dc577b1d171e0e298f65a3,2716028,https://www.airbnb.com/rooms/2716028,20250927041820,2025-09-28,city scrape,Reforma a la puerta,View of Reforma and the Angel of Indpendence. ...,NaN,https://a0.muscache.com/pictures/35451965/0304...,...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Tabla de contingencia (filas nulas en cada columna):


reviews_per_month_nulo,False,True
last_review_nulo,,
False,23650,0
True,0,3401



¿Siempre coinciden los nulos? True


### Formato de `price` y valores negativos

Antes de transformar, se revisa si `price` llega como texto con símbolos o separadores inconsistentes, y si hay columnas numéricas con valores negativos inesperados.

In [13]:
res_price = analizar_campo_price(df_listings, col="price", nombre_coleccion="LISTINGS")
res_neg = analizar_valores_negativos(
    df_listings,
    columnas=["price", "minimum_nights", "number_of_reviews", "calculated_host_listings_count", "availability_365", "number_of_reviews_ltm"],
    nombre_coleccion="LISTINGS",
)


ANÁLISIS DE CAMPO `price` - LISTINGS
--------------------------------------------------------------------------------
   Tipo pandas: str
   Registros: 27,051 | Nulos: 3,484 (12.88%)

   pd.to_numeric(errors='coerce'):
      - Reconocidos como número: 0
      - No nulos que NO convierten: 23,567

   Formato de la representación en texto (no nulos):


,conteo
price,
simbolo_moneda_en_texto,23567


   Filas con símbolo $ € £ ¥ ¢ en el texto: 23,567

   Ejemplos que no convierten a número (hasta 8):


,price
0,"$3,673.00"
1,"$18,000.00"
2,$591.00
7,$321.00
8,"$1,190.00"
12,$768.00
13,$905.00
14,"$1,056.00"



VALORES NEGATIVOS - LISTINGS
--------------------------------------------------------------------------------

   Columna: price
      - Numéricos válidos: 0 | No nulos no convertibles: 23,567
      - Negativos: 0 | Ceros: 0 | Positivos: 0
      >>> Sin valores negativos en valores numéricos válidos.

   Columna: minimum_nights
      - Numéricos válidos: 27,051 | No nulos no convertibles: 0
      - Negativos: 0 | Ceros: 0 | Positivos: 27,051
      - Mínimo: 1 | Máximo: 1125
      >>> Sin valores negativos en valores numéricos válidos.

   Columna: number_of_reviews
      - Numéricos válidos: 27,051 | No nulos no convertibles: 0
      - Negativos: 0 | Ceros: 3,401 | Positivos: 23,650
      - Mínimo: 0 | Máximo: 1434
      >>> Sin valores negativos en valores numéricos válidos.

   Columna: calculated_host_listings_count
      - Numéricos válidos: 27,051 | No nulos no convertibles: 0
      - Negativos: 0 | Ceros: 0 | Positivos: 27,051
      - Mínimo: 1 | Máximo: 221
      >>> Sin valore

### Interpretación del resultado

- Las filas cuyo price sea nulo, deben de ser descartadas ya que el precio no es un valor que se pueda inferir o calcular con base en otros datos. Es un valor que el anfitrión asigna a su propiedad con base en su criterio
- Dado que last_review y reviews_per_month tienen el mismo número de nulos, se realiza un análisis para comprobar que si uno de los valores es nulo el otro también. Siendo esto algo real.

Para este caso lo que haremos será obtener estos valores desde la colección de Reviews:
- Relacionando vía Listing_id, obtendremos:
- La fecha de la ultima Review
- El promedio de Reviews por mes

En caso de que luego de esto, hayan filas en donde el valor de last_review o de reviews_per_month sea nulo. Estás filas serán eliminadas

- Para los host_names sin valor, se asignará el valor de "empty_hostname" para agruparlos de mejor manera

### Análisis de duplicados

In [14]:
analizar_duplicados(df_listings, "LISTINGS", col_id='id')


REGISTROS DUPLICADOS - LISTINGS:
--------------------------------------------------------------------------------
   1. POR COLUMNA ID (id):
      - Total de registros: 27,051
      - Registros unicos: 27,051
      - Duplicados por ID: 0 (0.00%)


TypeError: unhashable type: 'list'

### Análisis de outliers

In [ ]:
analizar_outliers(df_listings, "LISTINGS", columnas=["price", "minimum_nights", "availability_365"])


ANÁLISIS DE OUTLIERS - LISTINGS:
--------------------------------------------------------------------------------

   price:
      - Mínimo: 61.00
      - Q1 (25%): 643.00
      - Mediana: 1039.00
      - Q3 (75%): 1611.00
      - Máximo: 900000.00
      - IQR: 968.00
      - Outliers detectados: 1,768 (7.50%)
      - Rango válido: [-809.00, 3063.00]

   minimum_nights:
      - Mínimo: 1.00
      - Q1 (25%): 1.00
      - Mediana: 2.00
      - Q3 (75%): 2.00
      - Máximo: 1125.00
      - IQR: 1.00
      - Outliers detectados: 3,429 (12.68%)
      - Rango válido: [-0.50, 3.50]

   availability_365:
      - Mínimo: 0.00
      - Q1 (25%): 140.00
      - Mediana: 269.00
      - Q3 (75%): 341.00
      - Máximo: 365.00
      - IQR: 201.00
      - Outliers detectados: 0 (0.00%)
      - Rango válido: [-161.50, 642.50]


### Interpretación del resultado

El resultado del análisis estádistico y de outliers, permite o nos da la opción de crear categorias para poder clasificar los listings.

### 1. price → Categoría de precio

**Estadísticos:**
- Q1: 643
- Mediana: 1039
- Q3: 1611
- Límite superior (sin outliers): 3063

**Nueva variable: `price_category`**

- Budget         → price ≤ 643
- Mid-range      → 643 < price ≤ 1039
- Upper-mid      → 1039 < price ≤ 1611
- Premium        → 1611 < price ≤ 3063
- Luxury/Outlier → price > 3063

---

### 2. minimum_nights → Tipo de estancia

**Estadísticos:**
- Q1: 1
- Mediana: 2
- Q3: 2
- Límite superior (sin outliers): 3.5

**Nueva variable: `stay_category`**

- Short stay     → minimum_nights ≤ 1
- Standard stay  → 1 < minimum_nights ≤ 2
- Extended stay  → 2 < minimum_nights ≤ 3
- Long stay/outlier → minimum_nights > 3

---

### 3. availability_365 → Nivel de disponibilidad

**Estadísticos:**
- Q1: 140
- Mediana: 269
- Q3: 341
- Sin outliers detectados

**Nueva variable: `availability_category`**

- Low availability    → availability_365 ≤ 140
- Medium availability → 140 < availability_365 ≤ 269
- High availability   → 269 < availability_365 ≤ 341
- Very high availability → availability_365 > 341

---

### Notas metodológicas

- Se utilizó el criterio de IQR (Interquartile Range) para detectar outliers.
- Las categorías se definieron con base en cuantiles para mantener balance en la distribución.
- Los valores fuera del rango superior se consideran "outliers" pero se mantienen como categoría separada para análisis.




## 2. Análisis de Colección: REVIEWS

Información sobre las reseñas que dejan los usuarios de los apartamentos.

### Análisis de estructura

In [ ]:
analizar_estructura(df_reviews, "REVIEWS")


COLECCIÓN: REVIEWS

PRIMERAS FILAS (primeros 5 registros):


,_id,listing_id,date
0,69dbd860dcaeaf06fe060c7e,44616,2011-11-09
1,69dbd860dcaeaf06fe060c7f,44616,2012-08-16
2,69dbd860dcaeaf06fe060c80,44616,2012-12-28
3,69dbd860dcaeaf06fe060c81,44616,2013-01-04
4,69dbd860dcaeaf06fe060c82,44616,2013-03-19



DIMENSIONES:
   - Filas: 1,454,740
   - Columnas: 3

TIPOS DE DATOS:
INFO
<class 'pandas.DataFrame'>
RangeIndex: 1454740 entries, 0 to 1454739
Columns: 3 entries, _id to date
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 33.3 MB
TYPES
_id                      str
listing_id             int64
date          datetime64[us]
dtype: object

ESTADÍSTICAS (Columnas numéricas):


,count,mean,min,25%,50%,75%,max,std
listing_id,1454740.0,355821508552638016.0,44616.0,26850859.0,46919861.0,791170307741018368.0,1517372885452894720.0,471299866764997760.0
date,1454740,2023-02-18 02:05:34.001128,2011-04-02 00:00:00,2022-02-02 00:00:00,2023-10-17 00:00:00,2024-11-18 00:00:00,2025-09-28 00:00:00,NaN



ESTADÍSTICAS COMPLETAS (incluye variables categóricas):


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
_id,1454740,1454740,69dbd860dcaeaf06fe060c7e,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
listing_id,1454740.0,NaN,NaN,NaN,355821508552638016.0,44616.0,26850859.0,46919861.0,791170307741018368.0,1517372885452894720.0,471299866764997760.0
date,1454740,NaN,NaN,NaN,2023-02-18 02:05:34.001128,2011-04-02 00:00:00,2022-02-02 00:00:00,2023-10-17 00:00:00,2024-11-18 00:00:00,2025-09-28 00:00:00,NaN



COLUMNAS NO NUMÉRICAS (2):
   - _id: 1454740 valores únicos
      Principales valores:
{'69dbd860dcaeaf06fe060c7e': 1, '69dbd860dcaeaf06fe060c7f': 1, '69dbd860dcaeaf06fe060c80': 1, '69dbd860dcaeaf06fe060c81': 1, '69dbd860dcaeaf06fe060c82': 1}
   - date: 4754 valores únicos
      Principales valores:
{Timestamp('2025-08-31 00:00:00'): 2753, Timestamp('2025-06-29 00:00:00'): 2657, Timestamp('2025-03-17 00:00:00'): 2542, Timestamp('2025-04-27 00:00:00'): 2520, Timestamp('2025-02-24 00:00:00'): 2510}


### Interpretación del resultado

- La colección Reviews (Reseñas) corresponde a las opiniones de la gente acerca del servicio obtenido
- La colección tiene la siguiente estructura: 
    - Filas: 1.454.740
    - Columnas: 3

*Explicación de sus campos*
- **_id (ObjectId)**  
  Identificador único del documento en MongoDB.  
  No tiene valor analítico directo.

- **listing_id (Int32)**  
  Identificador del alojamiento al que pertenece la reseña.  
  Permite relacionar esta colección con `Listings` y `Calendar`.

- **date (Date)**  
  Fecha en la que se realizó la reseña.  
  Es clave para análisis de tendencia, estacionalidad y actividad en el tiempo.

**Consideraciones**

- No todas las reservas generan una reseña → los datos pueden subestimar la demanda real
- No incluye contenido textual de la reseña (solo metadata)
- Puede haber sesgo hacia usuarios más activos o experiencias extremas

### Análisis de valores nulos

In [ ]:
analizar_valores_nulos(df_reviews, "REVIEWS")


VALORES NULOS Y FALTANTES - REVIEWS:
--------------------------------------------------------------------------------
   Todas las columnas están completas (sin nulos)


### Análisis de duplicados

In [ ]:
analizar_duplicados(df_reviews, "REVIEWS", col_id='_id')


REGISTROS DUPLICADOS - REVIEWS:
--------------------------------------------------------------------------------
   1. POR COLUMNA ID (_id):
      - Total de registros: 1,454,740
      - Registros únicos: 1,454,740
      - Duplicados por ID: 0 (0.00%)

   2. FILAS COMPLETAMENTE DUPLICADAS (todas las columnas):
      - Filas duplicadas: 0 (0.00%)

   Excelente: No hay duplicados en esta colección


### Análisis de outliers

In [ ]:
analizar_outliers(df_reviews, "REVIEWS")


ANÁLISIS DE OUTLIERS - REVIEWS:
--------------------------------------------------------------------------------

   listing_id:
      - Mínimo: 44616.00
      - Q1 (25%): 26850859.00
      - Mediana: 46919861.00
      - Q3 (75%): 791170307741018368.00
      - Máximo: 1517372885452894720.00
      - IQR: 791170307714167552.00
      - Outliers detectados: 0 (0.00%)
      - Rango válido: [-1186755461544400384.00, 1977925769312269568.00]


## 3. Análisis de Colección: CALENDAR

Información sobre disponibilidad, precios y políticas de noches mínimas para cada día.

### Análisis de estructura

In [ ]:
analizar_estructura(df_calendar, "CALENDAR")


COLECCIÓN: CALENDAR

PRIMERAS FILAS (primeros 5 registros):


,_id,listing_id,date,available,minimum_nights,maximum_nights
0,69dbd65cdcaeaf06fe6f63a4,7860479,2025-09-28,False,5,28
1,69dbd65cdcaeaf06fe6f63a5,7860479,2025-09-29,False,5,28
2,69dbd65cdcaeaf06fe6f63a6,7860479,2025-09-30,False,5,28
3,69dbd65cdcaeaf06fe6f63a7,7860479,2025-10-01,False,5,28
4,69dbd65cdcaeaf06fe6f63a8,7860479,2025-10-02,False,5,28



DIMENSIONES:
   - Filas: 9,873,624
   - Columnas: 6

TIPOS DE DATOS:
INFO
<class 'pandas.DataFrame'>
RangeIndex: 9873624 entries, 0 to 9873623
Columns: 6 entries, _id to maximum_nights
dtypes: bool(1), datetime64[us](1), int64(3), str(1)
memory usage: 386.1 MB
TYPES
_id                          str
listing_id                 int64
date              datetime64[us]
available                   bool
minimum_nights             int64
maximum_nights             int64
dtype: object

ESTADÍSTICAS (Columnas numéricas):


,count,mean,min,25%,50%,75%,max,std
listing_id,9873624.0,700355870257449216.0,35797.0,44104681.0,830736761677326976.0,1217411034178283008.0,1518561444972990720.0,570260905803648384.0
date,9873624,2026-03-28 17:33:26.990745,2025-09-27 00:00:00,2025-12-27 00:00:00,2026-03-29 00:00:00,2026-06-28 00:00:00,2026-09-27 00:00:00,NaN
minimum_nights,9873624.0,4.309379,1.0,1.0,2.0,2.0,1125.0,23.623162
maximum_nights,9873624.0,794562.366399,1.0,365.0,730.0,1125.0,2147483647.0,41281702.330175



ESTADÍSTICAS COMPLETAS (incluye variables categóricas):


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
_id,9873624,9873624,69dbd65cdcaeaf06fe6f63a4,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
listing_id,9873624.0,NaN,NaN,NaN,700355870257449216.0,35797.0,44104681.0,830736761677326976.0,1217411034178283008.0,1518561444972990720.0,570260905803648384.0
date,9873624,NaN,NaN,NaN,2026-03-28 17:33:26.990745,2025-09-27 00:00:00,2025-12-27 00:00:00,2026-03-29 00:00:00,2026-06-28 00:00:00,2026-09-27 00:00:00,NaN
available,9873624,2,True,6299554,NaN,NaN,NaN,NaN,NaN,NaN,NaN
minimum_nights,9873624.0,NaN,NaN,NaN,4.309379,1.0,1.0,2.0,2.0,1125.0,23.623162
maximum_nights,9873624.0,NaN,NaN,NaN,794562.366399,1.0,365.0,730.0,1125.0,2147483647.0,41281702.330175



COLUMNAS NO NUMÉRICAS (3):
   - _id: 9873624 valores únicos
      Principales valores:
{'69dbd65cdcaeaf06fe6f63a4': 1, '69dbd65cdcaeaf06fe6f63a5': 1, '69dbd65cdcaeaf06fe6f63a6': 1, '69dbd65cdcaeaf06fe6f63a7': 1, '69dbd65cdcaeaf06fe6f63a8': 1}
   - date: 366 valores únicos
      Principales valores:
{Timestamp('2025-09-28 00:00:00'): 27051, Timestamp('2025-09-29 00:00:00'): 27051, Timestamp('2025-09-30 00:00:00'): 27051, Timestamp('2025-10-01 00:00:00'): 27051, Timestamp('2025-10-02 00:00:00'): 27051}
   - available: 2 valores únicos
      Principales valores:
{True: 6299554, False: 3574070}


### Interpretación del resultado

- La colección Calendar (Calendario) corresponde a la disponibilidad diaria de un alojamiento. Cada fila es un día específico de una propiedad
- La colección tiene la siguiente estructura: 
    - Filas: 9,873,624
    - Columnas: 6

*Explicación de sus campos*
- **_id (ObjectId)**  
  Identificador único del documento en MongoDB.  
  No tiene valor analítico directo.

- **listing_id (Int32)**  
  Identificador del alojamiento.  
  Permite relacionar esta colección con `Listings`.

- **date (Date)**  
  Fecha específica del calendario.  
  Cada registro corresponde a un día para un alojamiento.

- **available (Boolean)**  
  Indica si el alojamiento está disponible en esa fecha:
  - `true` → disponible para reservar  
  - `false` → no disponible (reservado o bloqueado)

- **minimum_nights (Int32)**  
  Número mínimo de noches requeridas para reservar en esa fecha.

- **maximum_nights (Int32)**  
  Número máximo de noches permitidas para una reserva.

**Consideraciones**

- `available = false` no siempre implica reserva (puede ser bloqueo manual del host)
- Requiere agregación para análisis (datos están a nivel diario)
- Puede ser voluminoso (una fila por día por propiedad)

### Interpretación general

Este dataset permite analizar la **disponibilidad y ocupación diaria de los alojamientos**, ya que:

- Cada fila representa un día específico
- Permite construir el calendario completo de cada propiedad

### Análisis de valores nulos

In [ ]:
analizar_valores_nulos(df_calendar, "CALENDAR")


VALORES NULOS Y FALTANTES - CALENDAR:
--------------------------------------------------------------------------------
   Todas las columnas están completas (sin nulos)


### Análisis de duplicados

In [ ]:
analizar_duplicados(df_calendar, "CALENDAR", col_id='_id')


REGISTROS DUPLICADOS - CALENDAR:
--------------------------------------------------------------------------------
   1. POR COLUMNA ID (_id):
      - Total de registros: 9,873,624
      - Registros únicos: 9,873,624
      - Duplicados por ID: 0 (0.00%)

   2. FILAS COMPLETAMENTE DUPLICADAS (todas las columnas):
      - Filas duplicadas: 0 (0.00%)

   Excelente: No hay duplicados en esta colección


### Análisis de outliers

In [ ]:
analizar_outliers(df_calendar, "CALENDAR", columnas=["maximum_nights", "minimum_nights"])


ANÁLISIS DE OUTLIERS - CALENDAR:
--------------------------------------------------------------------------------

   maximum_nights:
      - Mínimo: 1.00
      - Q1 (25%): 365.00
      - Mediana: 730.00
      - Q3 (75%): 1125.00
      - Máximo: 2147483647.00
      - IQR: 760.00
      - Outliers detectados: 7,280 (0.07%)
      - Rango válido: [-775.00, 2265.00]

   minimum_nights:
      - Mínimo: 1.00
      - Q1 (25%): 1.00
      - Mediana: 2.00
      - Q3 (75%): 2.00
      - Máximo: 1125.00
      - IQR: 1.00
      - Outliers detectados: 1,296,995 (13.14%)
      - Rango válido: [-0.50, 3.50]


### Análisis de valores negativos

In [ ]:
analizar_valores_negativos(
    df_calendar,
    columnas=["minimum_nights", "maximum_nights"],
    nombre_coleccion="CALENDAR",
)



VALORES NEGATIVOS - CALENDAR
--------------------------------------------------------------------------------

   Columna: minimum_nights
      - Numéricos válidos: 9,873,624 | No nulos no convertibles: 0
      - Negativos: 0 | Ceros: 0 | Positivos: 9,873,624
      - Mínimo: 1 | Máximo: 1125
      >>> Sin valores negativos en valores numéricos válidos.

   Columna: maximum_nights
      - Numéricos válidos: 9,873,624 | No nulos no convertibles: 0
      - Negativos: 0 | Ceros: 0 | Positivos: 9,873,624
      - Mínimo: 1 | Máximo: 2147483647
      >>> Sin valores negativos en valores numéricos válidos.


{'minimum_nights': {'n_negativos': 0,
  'n_validos': 9873624,
  'min': 1.0,
  'max': 1125.0,
  'requiere_atencion': False},
 'maximum_nights': {'n_negativos': 0,
  'n_validos': 9873624,
  'min': 1.0,
  'max': 2147483647.0,
  'requiere_atencion': False}}

## 4. Visualizaciones y hallazgos principales

Las siguientes gráficas se construyen con variables alineadas con el taller y con mayor valor analático para inteligencia de negocios: `price`, `room_type`, `neighbourhood`, `minimum_nights`, fechas de `Reviews` y disponibilidad de `Calendar`.

### Preparación de datos para visualización

Para mejorar la legibilidad de algunas gráficas, se filtran valores nulos en `price`, se limita la visualización del precio al percentil 99 y se generan agregaciones mensuales para `Reviews` y `Calendar`.

In [ ]:
df_listings_viz = df_listings.copy()

# Forzar columnas clave a tipo numérico para evitar errores si llegan como texto.
df_listings_viz['price_num'] = pd.to_numeric(df_listings_viz['price'], errors='coerce')
df_listings_viz['minimum_nights_num'] = pd.to_numeric(df_listings_viz['minimum_nights'], errors='coerce')
df_listings_viz['availability_365_num'] = pd.to_numeric(df_listings_viz['availability_365'], errors='coerce')
df_listings_viz['number_of_reviews_num'] = pd.to_numeric(df_listings_viz['number_of_reviews'], errors='coerce')

df_listings_viz = df_listings_viz.dropna(subset=['price_num'])

price_q99 = df_listings_viz['price_num'].quantile(0.99)
df_price_plot = df_listings_viz[df_listings_viz['price_num'] <= price_q99].copy()

top_neighbourhoods = df_listings['neighbourhood'].value_counts().head(10)
avg_price_by_room_type = (
    df_listings_viz.groupby('room_type', as_index=False)['price_num']
    .mean()
    .sort_values('price_num', ascending=False)
)

reviews_by_month = df_reviews['date'].dt.to_period('M').value_counts().sort_index()
reviews_by_month.index = reviews_by_month.index.to_timestamp()

availability_by_month = (
    df_calendar.groupby(df_calendar['date'].dt.to_period('M'))['available']
    .mean()
    .sort_index()
    * 100
)
availability_by_month.index = availability_by_month.index.to_timestamp()

stay_labels = ['Short stay', 'Standard stay', 'Extended stay', 'Long stay/outlier']
stay_category = pd.cut(
    pd.to_numeric(df_listings['minimum_nights'], errors='coerce'),
    bins=[-float('inf'), 1, 2, 3, float('inf')],
    labels=stay_labels
)
stay_counts = stay_category.value_counts().reindex(stay_labels)

print(f'Percentil 99 de price: {price_q99:,.0f}')
display(avg_price_by_room_type)
display(top_neighbourhoods.to_frame('listings'))

### 4.1 Distribución del precio por noche

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df_price_plot['price_num'], bins=40, kde=True, ax=axes[0], color='#2a9d8f')
axes[0].set_title('Distribución del precio por noche (sin el 1% superior)')
axes[0].set_xlabel('Precio por noche (MXN)')
axes[0].set_ylabel('Frecuencia')

sns.boxplot(x=df_price_plot['price_num'], ax=axes[1], color='#e9c46a')
axes[1].set_title('Boxplot de precio por noche (sin el 1% superior)')
axes[1].set_xlabel('Precio por noche (MXN)')

plt.tight_layout()
plt.show()

### Interpretación

La distribución del precio es claramente asimétrica hacia la derecha, lo que indica que la mayor parte de los alojamientos se concentra en rangos bajos y medios, mientras que existe un grupo más pequeño con precios muy altos. Esto coincide con el análisis de outliers realizado previamente y justifica la creación de categorías de precio para la fase de transformación. Desde inteligencia de negocios, esta gráfica permite identificar que el mercado no es homogéneo y que conviene segmentarlo por niveles de precio en lugar de analizarlo con un único promedio global.

### 4.2 Precio promedio por tipo de habitación

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=avg_price_by_room_type, x='room_type', y='price_num', palette='Set2')
ax.set_title('Precio promedio por tipo de habitación')
ax.set_xlabel('Tipo de habitación')
ax.set_ylabel('Precio promedio (MXN)')
plt.xticks(rotation=10)

for patch in ax.patches:
    height = patch.get_height()
    ax.annotate(f'{height:,.0f}',
                (patch.get_x() + patch.get_width() / 2, height),
                ha='center', va='bottom', fontsize=9, xytext=(0, 5),
                textcoords='offset points')

plt.tight_layout()
plt.show()

### Interpretación

El precio promedio cambia de forma importante según el tipo de habitación. Este comportamiento muestra que `room_type` es una variable explicativa clave para segmentar la oferta y entender diferencias de valor dentro de la plataforma. Para el análisis de negocio, esta gráfica permite comparar rápidamente qué modalidad concentra los precios más altos y cuáles atienden segmentos más accesibles del mercado.

### 4.3 Top 10 barrios por cantidad de alojamientos

In [ ]:
plt.figure(figsize=(11, 6))
ax = sns.barplot(x=top_neighbourhoods.values, y=top_neighbourhoods.index, palette='Blues_r')
ax.set_title('Top 10 barrios por cantidad de alojamientos')
ax.set_xlabel('Cantidad de alojamientos')
ax.set_ylabel('Barrio')

for patch in ax.patches:
    width = patch.get_width()
    ax.annotate(f'{int(width):,}',
                (width, patch.get_y() + patch.get_height() / 2),
                ha='left', va='center', fontsize=9, xytext=(5, 0),
                textcoords='offset points')

plt.tight_layout()
plt.show()

### Interpretación

La oferta de alojamientos no se distribuye de manera uniforme entre los barrios. Existen zonas con una concentración mucho mayor de anuncios, lo que sugiere polos de mayor actividad turística o comercial dentro de la ciudad. Esta visualización es útil para identificar mercados más competidos y también para contextualizar posteriores comparaciones de precio, reseñas y disponibilidad por zona.

### 4.4 Evolución mensual de reseñas

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(x=reviews_by_month.index, y=reviews_by_month.values, marker='o', color='#e76f51')
plt.title('Cantidad de reseñas por mes')
plt.xlabel('Mes')
plt.ylabel('Número de reseñas')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Interpretación

La serie temporal de reseñas permite observar la dinámica de actividad en la plataforma a lo largo del tiempo. Los picos y caídas mensuales pueden asociarse con estacionalidad, periodos vacacionales o cambios en la demanda. En términos de inteligencia de negocios, esta gráfica ayuda a detectar meses de mayor interacción de usuarios y sirve como base para comparaciones posteriores con disponibilidad y ocupación.

### 4.5 Disponibilidad mensual promedio

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(x=availability_by_month.index, y=availability_by_month.values, marker='o', color='#264653')
plt.axhline(availability_by_month.mean(), linestyle='--', color='gray', label=f'Promedio: {availability_by_month.mean():.1f}%')
plt.title('Porcentaje promedio de disponibilidad mensual')
plt.xlabel('Mes')
plt.ylabel('Disponibilidad promedio (%)')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

### Interpretación

La disponibilidad mensual promedio muestra cómo cambia la oferta efectiva de noches disponibles a lo largo del tiempo. Un porcentaje alto puede asociarse con menor ocupación, mientras que un porcentaje más bajo puede sugerir mayor uso o reserva de los alojamientos. Esta gráfica aporta una visión temporal importante para el análisis de estacionalidad y para futuras métricas de ocupación.

### 4.6 Distribución de categorías de estancia mínima

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.barplot(x=stay_counts.index, y=stay_counts.values, palette='pastel')
ax.set_title('Distribución de categorías de estancia mínima')
ax.set_xlabel('Categoría de estancia')
ax.set_ylabel('Cantidad de alojamientos')
plt.xticks(rotation=10)

for patch in ax.patches:
    height = patch.get_height()
    ax.annotate(f'{int(height):,}',
                (patch.get_x() + patch.get_width() / 2, height),
                ha='center', va='bottom', fontsize=9, xytext=(0, 5),
                textcoords='offset points')

plt.tight_layout()
plt.show()

### Interpretación

La mayoría de los alojamientos se concentra en categorías de estancia corta o estándar, mientras que una menor proporción exige estancias más largas. Este hallazgo es consistente con el análisis de outliers en `minimum_nights` y refuerza la utilidad de crear una variable categórica para distinguir comportamientos normales de políticas más restrictivas. Desde el punto de vista del negocio, esta segmentación ayuda a diferenciar alojamientos orientados a turismo corto frente a estancias prolongadas.

## Conclusiones finales del EDA

### Inconsistencias detectadas

- **Listings — `price` nulo:** hay anuncios sin precio; no es imputable de forma fiable y esas filas deben excluirse del análisis o de la capa transformada.
- **Listings — `last_review` y `reviews_per_month`:** comparten el mismo patrón de nulos; donde falta uno suele faltar el otro. Conviene completar o recalcular desde **Reviews** y descartar lo que siga incompleto si no aporta negocio.
- **Listings — `host_name` ausente:** pocos registros sin nombre de anfitrión; conviene un valor sustituto explícito para no mezclar nulos con texto en reportes.
- **Listings — `number_of_reviews` vs conteo real en Reviews:** el valor agregado en el anuncio puede no coincidir con el número de filas en `Reviews` por corte temporal, snapshots o reglas de la plataforma; la transformación debe alinear ambas fuentes con una regla explícita (ver abajo).
- **Listings — valores extremos:** `price`, `minimum_nights` y en menor medida otras numéricas presentan colas pesadas y outliers por IQR; no son errores automáticos, pero motivan **categorías** o tratamiento aparte en modelos.
- **Reviews / Calendar — escala y granularidad:** Reviews está a nivel reseña y Calendar a nivel día × listing; cualquier métrica global debe documentar si se agrega, filtra por fechas o se deja en detalle.

### Correlaciones relevantes

- Este EDA se centró en **calidad, distribuciones y outliers**, no en una matriz de correlación entre todas las variables numéricas.
- La relación estructural más importante es **`listing_id` / `id`** entre colecciones: permite unir Listings con Reviews y Calendar para métricas de actividad, disponibilidad y coherencia de conteos.
- Las variables de **fecha** (`last_review`, fechas en Reviews, `date` en Calendar) habilitan estudios temporales y estacionalidad una vez homogeneizadas en la fase de transformación.
- Si el curso o el modelo lo requieren, se puede añadir un bloque dedicado (por ejemplo correlación de Spearman/Pearson en Listings ya limpios) sobre `price`, `availability_365`, `number_of_reviews`, etc.

### Decisiones y transformaciones definidas por Colección

**Listings**

- **Eliminar** registros con `price` nulo.
- **`last_review` y `reviews_per_month`:** obtener o recalcular desde **Reviews** (última fecha y promedio mensual por listing); **eliminar** filas que sigan con nulos en esos campos si no se justifica su retención.
- **`host_name` nulo:** imputar literal `empty_hostname` (o equivalente acordado) para análisis y agrupaciones.
- **Variables derivadas por cuantiles (EDA):** `price_category`, `stay_category` (desde `minimum_nights`), `availability_category` (desde `availability_365`), según los cortes documentados en la interpretación de outliers.
- **`number_of_reviews` alineado con Reviews:** para cada listing, calcular el conteo de reseñas en la colección Reviews. Regla propuesta:
  - Si en tabla `number_of_reviews == 0` y el conteo calculado es distinto de 0, **asignar el conteo calculado**.
  - Si el valor en tabla es **mayor** que el conteo calculado, **no cambiar** (mantener el valor de tabla).
  - Si el valor en tabla es **menor** que el conteo calculado, **conservar el valor de tabla** (no sustituir por el conteo mayor; la tabla prevalece frente al agregado cuando el anuncio ya trae un número positivo).
  - En conjunto: con valor en tabla **> 0** se mantiene siempre el dato del listing; con **0** se puede actualizar desde el conteo cuando este demuestre reseñas existentes.
- **Formato y limpieza numérica:** validar `price` (y otras columnas) según el análisis de formato y de valores negativos; en los datos actuales el precio ya aparece como numérico, pero la regla sirve si el origen cambia.

**Reviews**

- Actúa como **fuente de verdad** para fechas de reseña y para el **conteo por listing** usado en la regla de `number_of_reviews` y en la imputación de `last_review` / `reviews_per_month`.
- Mantener la trazabilidad por `listing_id` (y `_id` de reseña) al agregar o filtrar; revisar duplicados y outliers según los resultados de las celdas de calidad.

**Calendar**

- Datos **diarios y voluminosos**; las transformaciones típicas pasan por agregaciones (por mes, semana o listing) o filtros de ventana temporal según el uso (ocupación, estacionalidad).
- Validar **`minimum_nights` y `maximum_nights`** (incluido el análisis de valores negativos); reglas de negocio pueden acotar rangos o marcar fechas anómalas.
- Cruzar con Listings por `listing_id` para enriquecer análisis de disponibilidad frente a precio y categorías definidas en Listings.

### Nota sobre validación posterior a las transformaciones

Tras aplicar las transformaciones acordadas (limpieza, imputaciones, reglas de negocio, derivación de variables, agregaciones, etc.), **conviene volver a ejecutar parte de las validaciones de este EDA** sobre los datos resultantes para **comprobar la efectividad** de los cambios. En la práctica eso puede incluir, según lo transformado: conteo de nulos por columna, duplicados, comprobación de rangos y valores negativos, coherencia entre colecciones (por ejemplo `number_of_reviews` vs Reviews, o fechas imputadas), y revisión de distribuciones u outliers en variables clave. El objetivo es confirmar que las reglas se aplicaron bien y que no aparecieron efectos secundarios (pérdida excesiva de filas, nuevos nulos, valores fuera de dominio).
